# 🔍 Network Scanner — Colab Edition
**Advanced Level — Project 12**

Skills: Socket Programming • Security

A TCP port scanner built with Python's `socket` module — no third-party network libraries required. It checks which ports are open on a target host, grabs service banners where available, and prints a readable report.

## ⚠️ Legal & ethical use — read first
Only scan hosts and networks **you own, or that you have explicit written permission to test**. Scanning systems you don't own or lack authorization for can violate laws such as the U.S. Computer Fraud and Abuse Act (and equivalents elsewhere) and most ISPs'/cloud providers' acceptable use policies, even if no damage occurs. This notebook defaults to `127.0.0.1` (your own machine) for that reason — treat changing the target as something you do only against infrastructure you're authorized to test (your own lab VM, a host you own, or an intentionally vulnerable practice target like a local `DVWA`/`Metasploitable` VM or a sanctioned CTF range).

**Note on Colab specifically:** Colab's sandboxed VM has its own network restrictions and shared IP ranges, so scanning arbitrary external hosts from Colab may also violate Google's terms of service. This notebook is best used against `localhost`, a host on a network you control, or services you deliberately spin up for testing.

## 1. Imports
Everything here is from Python's standard library — no installs needed.

In [10]:
import socket
import concurrent.futures
import ipaddress
import time
from datetime import datetime

## 2. Authorization gate
A simple guard so the scanner refuses to run until you explicitly confirm you're authorized against the target. This doesn't replace your own judgment or legal obligations — it's just a speed bump against accidental misuse.

In [11]:
def confirm_authorization(target):
    print(f"Target: {target}")
    ans = input("Do you own this host, or have explicit written permission to scan it? (yes/no): ").strip().lower()
    if ans != "yes":
        raise PermissionError("Scan aborted — authorization not confirmed.")
    print("Authorization confirmed. Proceeding.\n")

## 3. Port lists
A small curated list of common ports, plus a helper to build custom ranges.

In [12]:
COMMON_PORTS = {
    21: 'FTP', 22: 'SSH', 23: 'Telnet', 25: 'SMTP', 53: 'DNS',
    80: 'HTTP', 110: 'POP3', 111: 'RPCbind', 135: 'MSRPC', 139: 'NetBIOS',
    143: 'IMAP', 443: 'HTTPS', 445: 'SMB', 993: 'IMAPS', 995: 'POP3S',
    1723: 'PPTP', 3306: 'MySQL', 3389: 'RDP', 5432: 'PostgreSQL',
    5900: 'VNC', 6379: 'Redis', 8080: 'HTTP-Alt', 8443: 'HTTPS-Alt',
    27017: 'MongoDB',
}

def port_range(start, end):
    return list(range(start, end + 1))

## 4. Core scanning logic
A TCP connect scan: for each port, attempt a real 3-way handshake with a short timeout. This is the same technique tools like `nmap -sT` use — slower than a raw SYN scan but doesn't need elevated privileges, which makes it Colab-friendly.

In [13]:
def scan_port(target_ip, port, timeout=0.5):
    """Attempt a TCP connection to a single port. Returns (port, is_open, banner)."""
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(timeout)
    banner = None
    try:
        result = sock.connect_ex((target_ip, port))
        is_open = (result == 0)
        if is_open:
            banner = grab_banner(sock)
    except socket.error:
        is_open = False
    finally:
        sock.close()
    return port, is_open, banner

def grab_banner(sock, timeout=0.5):
    """Try to read a short greeting/banner from an already-connected socket."""
    try:
        sock.settimeout(timeout)
        data = sock.recv(128)
        return data.decode(errors='ignore').strip() or None
    except Exception:
        return None

## 5. Concurrent scan runner
Scans many ports in parallel with a thread pool instead of one at a time, which is what makes scanning a full port range practical.

In [14]:
def scan_target(target, ports, timeout=0.5, max_workers=100):
    try:
        target_ip = str(ipaddress.ip_address(target))
    except ValueError:
        target_ip = socket.gethostbyname(target)  # resolve hostname to IP

    print(f"Scanning {target} ({target_ip}) — {len(ports)} ports — started {datetime.now().strftime('%H:%M:%S')}\n")
    start = time.time()
    open_ports = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(scan_port, target_ip, p, timeout): p for p in ports}
        for future in concurrent.futures.as_completed(futures):
            port, is_open, banner = future.result()
            if is_open:
                open_ports.append((port, banner))

    elapsed = time.time() - start
    open_ports.sort()

    print(f"Scan finished in {elapsed:.2f}s — {len(open_ports)} open port(s) found\n")
    print(f"{'PORT':<8}{'SERVICE':<12}{'BANNER'}")
    print('-' * 50)
    for port, banner in open_ports:
        service = COMMON_PORTS.get(port, 'unknown')
        print(f"{port:<8}{service:<12}{banner or ''}")

    return open_ports

## 6. Run a scan
Defaults to `127.0.0.1` (your own Colab VM) and the common-ports list. Change `TARGET` only for hosts you're authorized to test.

In [15]:
TARGET = "127.0.0.1"

confirm_authorization(TARGET)
results = scan_target(TARGET, list(COMMON_PORTS.keys()), timeout=0.5)

Target: 127.0.0.1
Do you own this host, or have explicit written permission to scan it? (yes/no): yes
Authorization confirmed. Proceeding.

Scanning 127.0.0.1 (127.0.0.1) — 24 ports — started 04:52:07

Scan finished in 0.51s — 1 open port(s) found

PORT    SERVICE     BANNER
--------------------------------------------------
8080    HTTP-Alt    


### Scan a custom port range instead

In [16]:
# TARGET = "127.0.0.1"
# confirm_authorization(TARGET)
# results = scan_target(TARGET, port_range(1, 1024), timeout=0.3)

## 7. Multi-host sweep (for a range/subnet you own)
Scans a small set of hosts for one specific port each — useful for a quick "who's alive and listening on port 80" style sweep across a lab subnet.

In [17]:
def sweep_hosts(hosts, port, timeout=0.5):
    print(f"Sweeping {len(hosts)} host(s) on port {port}\n")
    alive = []
    for host in hosts:
        _, is_open, _ = scan_port(host, port, timeout)
        status = "OPEN" if is_open else "closed/filtered"
        print(f"{host:<16} port {port}: {status}")
        if is_open:
            alive.append(host)
    return alive

# Example — only against hosts you own/control:
# hosts = [str(ip) for ip in ipaddress.ip_network('192.168.1.0/29').hosts()]
# sweep_hosts(hosts, port=80)

## 8. Export results

In [18]:
import json

def save_report(target, results, filename='scan_report.json'):
    report = {
        'target': target,
        'scanned_at': datetime.now().isoformat(),
        'open_ports': [
            {'port': p, 'service': COMMON_PORTS.get(p, 'unknown'), 'banner': b}
            for p, b in results
        ],
    }
    with open(filename, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"Report saved to {filename}")

save_report(TARGET, results)

Report saved to scan_report.json


## Notes
- This is a **TCP connect scan**, which fully completes the handshake — reliable and needs no elevated privileges, but is more visible to logging/IDS than a stealth SYN scan.
- Closed ports return quickly; filtered ports (blocked by a firewall) will usually just time out, which is why `timeout` matters for scan speed.
- Banner grabbing only works for services that send a greeting immediately on connect (e.g. FTP, SSH); many services wait for the client to speak first and won't show a banner here.
- For real-world security assessments, a purpose-built tool like `nmap` (with proper authorization) is far more capable — this notebook is meant as a learning exercise in socket programming and how port scanning works under the hood, not a replacement for professional tooling.